# Step 3 — Week 4: Incremental Indexing, Caching, and Reranking

## Goals
- Handle larger corpora with **incremental indexing**
- Add **embedding caching** and **batch ingestion**
- Introduce **reranking** for better relevance

This notebook keeps everything local and lightweight so you can run it quickly.

In [1]:
import os
import re
import json
import math
import time
import sqlite3
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

### Concept: Why this setup scales
- `HashingVectorizer` is **stateless**, so new chunks can be embedded without refitting a model.
- We persist index artifacts locally (`.npz` + `.csv`) so indexing can resume across runs.
- We cache embeddings by content hash to skip recomputing unchanged text.

In [2]:
# Workspace paths
BASE_DIR = Path.cwd()
WEEK4_DIR = BASE_DIR / "week4_artifacts"
WEEK4_DIR.mkdir(parents=True, exist_ok=True)

DOC_PATH = WEEK4_DIR / "documents_v1.jsonl"
DOC_PATH_V2 = WEEK4_DIR / "documents_v2.jsonl"
INDEX_MATRIX_PATH = WEEK4_DIR / "index_vectors.npz"
INDEX_META_PATH = WEEK4_DIR / "index_metadata.csv"
CACHE_DB_PATH = WEEK4_DIR / "embedding_cache.sqlite"

print("Artifacts folder:", WEEK4_DIR)

Artifacts folder: c:\Users\lolen\Downloads\Jupyter Mac\Python\2-AI\LLM\LLM-Repo\Topics\AI_Eng_Path\Step_3_Retrieval_Systems_RAG\week4_artifacts


### Concept: Simulating corpus growth
We create two corpus snapshots:
1. **v1**: initial documents
2. **v2**: one updated document + one new document

Incremental indexing should only process changed/new chunks in v2.

In [3]:
docs_v1 = [
    {"doc_id": "doc_001", "title": "RAG Basics", "text": "RAG combines retrieval and generation. Chunk quality and top-k strongly affect final answer quality."},
    {"doc_id": "doc_002", "title": "Vector Databases", "text": "Vector databases store embeddings for semantic search. Metadata filtering reduces search space."},
    {"doc_id": "doc_003", "title": "Evaluation", "text": "Retrieval evaluation often uses hit rate at k and mean reciprocal rank."},
]

docs_v2 = [
    {"doc_id": "doc_001", "title": "RAG Basics", "text": "RAG combines retrieval and generation. Good chunking, overlap tuning, and reranking can improve grounded answers."},
    {"doc_id": "doc_002", "title": "Vector Databases", "text": "Vector databases store embeddings for semantic search. Metadata filtering reduces search space."},
    {"doc_id": "doc_003", "title": "Evaluation", "text": "Retrieval evaluation often uses hit rate at k and mean reciprocal rank."},
    {"doc_id": "doc_004", "title": "Reranking", "text": "Reranking refines top candidates from first-stage retrieval and improves precision in final context selection."},
]

def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(DOC_PATH, docs_v1)
write_jsonl(DOC_PATH_V2, docs_v2)
print("Wrote", DOC_PATH.name, "and", DOC_PATH_V2.name)

Wrote documents_v1.jsonl and documents_v2.jsonl


### Concept: Incremental indexing design
- Chunk each document into smaller units for retrieval.
- Build a `chunk_hash` from normalized text.
- Before embedding, check if the hash already exists in the index metadata.
- If it exists, skip; if not, embed + append.

This avoids reindexing the full corpus every time.

In [5]:
@dataclass
class ChunkRecord:
    chunk_id: str
    doc_id: str
    title: str
    text: str
    chunk_hash: str

def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text.strip().lower())
    return text

def hash_text(text: str) -> str:
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()

def simple_chunk_text(text: str, chunk_words: int = 20, overlap_words: int = 5) -> List[str]:
    words = text.split()
    if not words:
        return []
    step = max(1, chunk_words - overlap_words)
    chunks = []
    for i in range(0, len(words), step):
        w = words[i:i + chunk_words]
        if not w:
            break
        chunks.append(" ".join(w))
        if i + chunk_words >= len(words):
            break
    return chunks

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def build_chunk_records(docs: List[Dict[str, Any]]) -> List[ChunkRecord]:
    out = []
    for doc in docs:
        parts = simple_chunk_text(doc["text"], chunk_words=20, overlap_words=5)
        for i, part in enumerate(parts):
            c_hash = hash_text(part)
            out.append(
                ChunkRecord(
                    chunk_id=f"{doc['doc_id']}_c{i}",
                    doc_id=doc["doc_id"],
                    title=doc["title"],
                    text=part,
                    chunk_hash=c_hash,
                )
            )
    return out

### Concept: Caching + batch ingestion
- **Cache**: if a chunk hash already has an embedding in SQLite, reuse it.
- **Batching**: transform many chunks together to reduce per-call overhead.
- **Persistence**: save vectors + metadata so future runs can continue incrementally.

In [6]:
class EmbeddingCache:
    def __init__(self, db_path: Path):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS embedding_cache (
                chunk_hash TEXT PRIMARY KEY,
                embedding BLOB NOT NULL
            )
        """)
        self.conn.commit()

    def get(self, chunk_hash: str):
        row = self.conn.execute("SELECT embedding FROM embedding_cache WHERE chunk_hash = ?", (chunk_hash,)).fetchone()
        if row is None:
            return None
        return np.frombuffer(row[0], dtype=np.float32)

    def set(self, chunk_hash: str, embedding: np.ndarray):
        vec = np.asarray(embedding, dtype=np.float32)
        self.conn.execute("INSERT OR REPLACE INTO embedding_cache(chunk_hash, embedding) VALUES (?, ?)", (chunk_hash, vec.tobytes()))

    def commit(self):
        self.conn.commit()

    def close(self):
        self.conn.close()

vectorizer = HashingVectorizer(n_features=1024, alternate_sign=False, norm="l2")

def embed_texts_batch(texts: List[str]) -> np.ndarray:
    # Dense float32 vectors for compact storage
    return vectorizer.transform(texts).toarray().astype(np.float32)

def load_existing_index() -> Tuple[np.ndarray, pd.DataFrame]:
    if INDEX_MATRIX_PATH.exists() and INDEX_META_PATH.exists():
        matrix = np.load(INDEX_MATRIX_PATH)["vectors"]
        meta = pd.read_csv(INDEX_META_PATH)
        return matrix, meta
    return np.empty((0, 1024), dtype=np.float32), pd.DataFrame(columns=["chunk_id", "doc_id", "title", "text", "chunk_hash", "indexed_at"])

def save_index(matrix: np.ndarray, meta: pd.DataFrame):
    np.savez_compressed(INDEX_MATRIX_PATH, vectors=matrix.astype(np.float32))
    meta.to_csv(INDEX_META_PATH, index=False)

def incremental_ingest(jsonl_path: Path, batch_size: int = 8) -> Dict[str, Any]:
    start = time.time()
    docs = load_jsonl(jsonl_path)
    chunks = build_chunk_records(docs)

    matrix, meta = load_existing_index()
    existing_hashes = set(meta["chunk_hash"].astype(str).tolist())

    to_add = [c for c in chunks if c.chunk_hash not in existing_hashes]

    cache = EmbeddingCache(CACHE_DB_PATH)
    new_vectors = []
    new_rows = []
    cache_hits = 0
    cache_misses = 0

    for i in range(0, len(to_add), batch_size):
        batch = to_add[i:i + batch_size]

        miss_items = []
        miss_texts = []
        vec_by_hash = {}

        for item in batch:
            cached = cache.get(item.chunk_hash)
            if cached is not None:
                vec_by_hash[item.chunk_hash] = cached
                cache_hits += 1
            else:
                miss_items.append(item)
                miss_texts.append(item.text)
                cache_misses += 1

        if miss_texts:
            miss_vecs = embed_texts_batch(miss_texts)
            for item, vec in zip(miss_items, miss_vecs):
                vec_by_hash[item.chunk_hash] = vec
                cache.set(item.chunk_hash, vec)

        for item in batch:
            vec = vec_by_hash[item.chunk_hash]
            new_vectors.append(vec)
            new_rows.append({
                "chunk_id": item.chunk_id,
                "doc_id": item.doc_id,
                "title": item.title,
                "text": item.text,
                "chunk_hash": item.chunk_hash,
                "indexed_at": pd.Timestamp.utcnow(),
            })

    cache.commit()
    cache.close()

    if new_vectors:
        add_mat = np.vstack(new_vectors).astype(np.float32)
        matrix = np.vstack([matrix, add_mat]) if len(matrix) else add_mat
        meta = pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)

    save_index(matrix, meta)

    return {
        "source": jsonl_path.name,
        "docs": len(docs),
        "total_chunks_seen": len(chunks),
        "new_chunks_indexed": len(to_add),
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "index_size": len(meta),
        "elapsed_sec": round(time.time() - start, 3),
    }

### Concept: First run vs incremental run
- First ingest on `v1` creates the initial index.
- Second ingest on `v2` should only add changed/new chunks.

This is the core behavior needed for large, evolving corpora.

In [7]:
# Optional reset for clean demo
for p in [INDEX_MATRIX_PATH, INDEX_META_PATH, CACHE_DB_PATH]:
    if p.exists():
        p.unlink()

run1 = incremental_ingest(DOC_PATH, batch_size=4)
run2 = incremental_ingest(DOC_PATH_V2, batch_size=4)

pd.DataFrame([run1, run2])

C:\Users\lolen\AppData\Local\Temp\ipykernel_20212\601455135.py:102: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  meta = pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)


,source,docs,total_chunks_seen,new_chunks_indexed,cache_hits,cache_misses,index_size,elapsed_sec
0,documents_v1.jsonl,3,3,3,0,3,3,0.080
1,documents_v2.jsonl,4,4,2,0,2,5,0.062


### Concept: Two-stage retrieval + reranking
- **Stage 1 (Retriever):** fast vector similarity to get top-k candidates.
- **Stage 2 (Reranker):** apply a richer relevance score on those candidates.

In production, stage 2 is often a cross-encoder. Here we use a lightweight lexical reranker to demonstrate the pattern.

In [8]:
def load_index_for_search() -> Tuple[np.ndarray, pd.DataFrame]:
    matrix = np.load(INDEX_MATRIX_PATH)["vectors"]
    meta = pd.read_csv(INDEX_META_PATH)
    return matrix, meta

def retrieve_top_k(query: str, k: int = 6) -> pd.DataFrame:
    matrix, meta = load_index_for_search()
    q_vec = embed_texts_batch([query])
    sims = cosine_similarity(q_vec, matrix).flatten()
    idx = np.argsort(-sims)[:k]
    out = meta.iloc[idx].copy()
    out["retrieval_score"] = sims[idx]
    return out.sort_values("retrieval_score", ascending=False)

def lexical_overlap_score(query: str, text: str) -> float:
    q_terms = set(re.findall(r"\w+", query.lower()))
    t_terms = set(re.findall(r"\w+", text.lower()))
    if not q_terms:
        return 0.0
    overlap = len(q_terms & t_terms) / len(q_terms)
    phrase_bonus = 0.15 if query.lower() in text.lower() else 0.0
    return min(1.0, overlap + phrase_bonus)

def rerank_candidates(query: str, candidates: pd.DataFrame, alpha: float = 0.7, beta: float = 0.3) -> pd.DataFrame:
    ranked = candidates.copy()
    ranked["rerank_score"] = ranked.apply(
        lambda r: alpha * float(r["retrieval_score"]) + beta * lexical_overlap_score(query, str(r["text"])),
        axis=1,
    )
    return ranked.sort_values("rerank_score", ascending=False)

### Concept: Compare before/after reranking
This final step shows how candidate ordering changes after reranking.

You should inspect whether top results become more aligned with the query intent.

In [9]:
query = "How does reranking improve final context selection in RAG?"

retrieved = retrieve_top_k(query, k=6)
reranked = rerank_candidates(query, retrieved, alpha=0.7, beta=0.3)

print("Top-3 before reranking")
display(retrieved[["doc_id", "title", "retrieval_score", "text"]].head(3))

print("Top-3 after reranking")
display(reranked[["doc_id", "title", "retrieval_score", "rerank_score", "text"]].head(3))

Top-3 before reranking


,doc_id,title,retrieval_score,text
4,doc_004,Reranking,0.430332,Reranking refines top candidates from first-stage retrieval and improves precision in final context selection.
3,doc_001,RAG Basics,0.242536,"RAG combines retrieval and generation. Good chunking, overlap tuning, and reranking can improve grounded answers."
0,doc_001,RAG Basics,0.157135,RAG combines retrieval and generation. Chunk quality and top-k strongly affect final answer quality.


Top-3 after reranking


,doc_id,title,retrieval_score,rerank_score,text
4,doc_004,Reranking,0.430332,0.467899,Reranking refines top candidates from first-stage retrieval and improves precision in final context selection.
3,doc_001,RAG Basics,0.242536,0.269775,"RAG combines retrieval and generation. Good chunking, overlap tuning, and reranking can improve grounded answers."
0,doc_001,RAG Basics,0.157135,0.176661,RAG combines retrieval and generation. Chunk quality and top-k strongly affect final answer quality.


## Week 4 Checklist
- [x] Incremental indexing with change detection (`chunk_hash`)
- [x] Embedding cache (SQLite)
- [x] Batch ingestion
- [x] Two-stage retrieval + reranking pattern

### Next upgrades
- Replace lexical reranker with a cross-encoder reranker
- Add metadata filters (time/source/topic) before reranking
- Track quality metrics (Hit@k, MRR, latency) per version